In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("data/raw/02_nav_history.csv")

In [3]:
df.head()

,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [4]:
df['date'] = pd.to_datetime(df['date'])

In [6]:
df = df.drop_duplicates(subset=['amfi_code', 'date'])

In [5]:
df = df.sort_values(by=['amfi_code', 'date'])

In [7]:
df['nav'] = df.groupby('amfi_code')['nav'].ffill()

In [8]:
df = df[df['nav'] > 0]

In [9]:
df.to_csv('cleaned_nav_history.csv', index=False)

Data is cleaned

2. Clean investor_transactions.csv — standardise transaction_type values (SIP/Lumpsum/Redemption), validate amount > 0, fix date formats, check KYC status enum values.

In [10]:
df_investor_transactions = pd.read_csv("data/raw/08_investor_transactions.csv")

In [11]:
df_investor_transactions.head()

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


In [12]:
df_investor_transactions['transaction_date'] = pd.to_datetime(df_investor_transactions['transaction_date'])

In [13]:
df_investor_transactions = df_investor_transactions[df_investor_transactions['amount_inr'] > 0]

In [14]:
print(df_investor_transactions['transaction_type'].unique())
print(df_investor_transactions['kyc_status'].unique())

['SIP' 'Redemption' 'Lumpsum']
['Verified' 'Pending']


In [15]:
allowed_kyc = ['Verified','Pending']

In [16]:
df_investor_transactions = df_investor_transactions[df_investor_transactions['kyc_status'].isin(allowed_kyc)]

In [17]:
df_investor_transactions.to_csv('cleaned_investor_transactions.csv', index=False)

Data Cleaned

3.Clean scheme_performance.csv — validate all return values are numeric, flag anomalies, check expense_ratio range (0.1% – 2.5%).

In [18]:
df_scheme_performance = pd.read_csv("data/raw/07_scheme_performance.csv")

In [19]:
df_scheme_performance.head()

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low


In [20]:
df_scheme_performance['return_1yr_pct'] = pd.to_numeric(df_scheme_performance['return_1yr_pct'], errors='coerce')
df_scheme_performance['return_3yr_pct'] = pd.to_numeric(df_scheme_performance['return_3yr_pct'], errors='coerce')
df_scheme_performance['return_5yr_pct'] = pd.to_numeric(df_scheme_performance['return_5yr_pct'], errors='coerce')

In [21]:
# This shows you the lowest and highest expense fees in your file
print("Lowest fee found:", df_scheme_performance['expense_ratio_pct'].min())
print("Highest fee found:", df_scheme_performance['expense_ratio_pct'].max())

Lowest fee found: 0.55
Highest fee found: 1.64


In [22]:
df_scheme_performance = df_scheme_performance[(df_scheme_performance['expense_ratio_pct'] >= 0.1) & (df_scheme_performance['expense_ratio_pct'] <= 2.5)]

In [24]:
anomalies_found = df_scheme_performance[(df_scheme_performance['return_1yr_pct'] > 100) | (df_scheme_performance['return_1yr_pct'] < -40)]

In [25]:
anomalies_found

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade


No anomalies found

In [26]:
df_scheme_performance.to_csv('cleaned_scheme_performance.csv', index=False)

Data Cleaned

4.Design SQLite star schema — write CREATE TABLE statements for dim_fund, dim_date, fact_nav, fact_transactions, fact_performance, fact_aum. Define primary and foreign keys.

In [27]:
import sqlite3

# This is the SQL script that creates your Star Schema structure.
# Notice that dim_fund and dim_date are placed FIRST.
schema_ddl = """
-- 1. TURN ON FOREIGN KEYS (Mandatory for SQLite)
PRAGMA foreign_keys = ON;

-- 2. CREATE DIMENSION TABLES FIRST
CREATE TABLE IF NOT EXISTS dim_fund (
    amfi_code TEXT PRIMARY KEY,
    fund_house TEXT NOT NULL,
    scheme_name TEXT NOT NULL,
    category TEXT,
    sub_category TEXT,
    plan TEXT,
    launch_date DATE,
    benchmark TEXT,
    expense_ratio_pct REAL,
    exit_load_pct REAL,
    min_sip_amount REAL,
    min_lumpsum_amount REAL,
    fund_manager TEXT,
    risk_category TEXT,
    sebi_category_code TEXT
);

CREATE TABLE IF NOT EXISTS dim_date (
    date DATE PRIMARY KEY,
    year INTEGER NOT NULL,
    quarter INTEGER NOT NULL,
    month INTEGER NOT NULL,
    month_name TEXT NOT NULL,
    day INTEGER NOT NULL,
    day_of_week TEXT NOT NULL,
    is_weekend INTEGER NOT NULL
);

-- 3. CREATE FACT TABLES SECOND (Because they depend on the Dimensions)
CREATE TABLE IF NOT EXISTS fact_nav (
    amfi_code TEXT,
    nav_date DATE,
    nav REAL NOT NULL,
    daily_return REAL,
    PRIMARY KEY (amfi_code, nav_date),
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code),
    FOREIGN KEY (nav_date) REFERENCES dim_date(date)
);

CREATE TABLE IF NOT EXISTS fact_transactions (
    transaction_id INTEGER PRIMARY KEY AUTOINCREMENT,
    investor_id TEXT NOT NULL,
    transaction_date DATE NOT NULL,
    amfi_code TEXT NOT NULL,
    transaction_type TEXT NOT NULL,
    amount_inr REAL NOT NULL,
    state TEXT,
    city TEXT,
    city_tier TEXT,
    age_group TEXT,
    gender TEXT,
    annual_income_lakh REAL,
    payment_mode TEXT,
    kyc_status TEXT,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code),
    FOREIGN KEY (transaction_date) REFERENCES dim_date(date)
);

CREATE TABLE IF NOT EXISTS fact_performance (
    amfi_code TEXT PRIMARY KEY,
    return_1yr_pct REAL,
    return_3yr_pct REAL,
    return_5yr_pct REAL,
    benchmark_3yr_pct REAL,
    alpha REAL,
    beta REAL,
    sharpe_ratio REAL,
    sortino_ratio REAL,
    std_dev_ann_pct REAL,
    max_drawdown_pct REAL,
    aum_crore REAL,
    expense_ratio_pct REAL,
    morningstar_rating INTEGER,
    risk_grade TEXT,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);

CREATE TABLE IF NOT EXISTS fact_aum (
    date DATE,
    fund_house TEXT,
    aum_lakh_crore REAL,
    aum_crore REAL,
    num_schemes INTEGER,
    PRIMARY KEY (date, fund_house),
    FOREIGN KEY (date) REFERENCES dim_date(date)
);
"""

def initialize_star_schema():
    # This automatically creates a file named 'mutual_fund_analytics.db' if it doesn't exist
    connection = sqlite3.connect("mutual_fund_analytics.db")
    cursor = connection.cursor()
    
    print("Initializing your Mutual Fund Database...")
    
    # Executes all of the SQL commands defined above sequentially
    cursor.executescript(schema_ddl)
    connection.commit()
    
    # Let's read back from SQLite to verify the tables are physically there
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()
    
    print("\n Success! Database generated with the following tables:")
    for row in tables:
        # Ignore internal sqlite tracking tables
        if "sqlite" not in row[0]:
            print(f" -> {row[0]}")
        
    connection.close()

if __name__ == "__main__":
    initialize_star_schema()

Initializing your Mutual Fund Database...

 Success! Database generated with the following tables:
 -> dim_fund
 -> dim_date
 -> fact_nav
 -> fact_transactions
 -> fact_performance
 -> fact_aum


In [28]:
import pandas as pd
from sqlalchemy import create_engine, text

# 1. Define file names and their corresponding database table names
data_mapping = {
    "dim_fund": "data/raw/01_fund_master.csv",  # (Optional parent master, included if you need it)
    "fact_nav": "cleaned_nav_history.csv",
    "fact_transactions": "cleaned_investor_transactions.csv",
    "fact_performance": "cleaned_scheme_performance.csv"
}

def load_and_verify_data():
    # 2. Initialize SQLAlchemy connection engine to the SQLite file
    engine = create_engine('sqlite:///bluestock_mf.db')
    
    print("--- STEP 1: LOADING DATASETS INTO SQLITE ---")
    
    csv_counts = {}
    
    for table_name, csv_file in data_mapping.items():
        try:
            # Load CSV into a Pandas DataFrame
            df = pd.read_csv(csv_file)
            
            # Record original CSV row count
            csv_counts[table_name] = len(df)
            
            # Upload data to SQLite table using df.to_sql()
            # if_exists='append' populates tables if you already created them with DDL
            # index=False avoids writing Pandas row index numbers as a separate column
            df.to_sql(table_name, con=engine, if_exists='append', index=False)
            print(f"Successfully loaded {csv_file} into table '{table_name}'.")
            
        except FileNotFoundError:
            print(f"Skipping {csv_file}: File not found in current folder.")
            continue

    print("\n--- STEP 2: VERIFYING ROW COUNTS ---")
    
    # 3. Connect to the database to extract loaded SQL row counts
    with engine.connect() as connection:
        for table_name in csv_counts.keys():
            try:
                # Run an explicit SQL COUNT query for confirmation
                query = text(f"SELECT COUNT(*) FROM {table_name};")
                sql_count = connection.execute(query).scalar()
                
                expected_count = csv_counts[table_name]
                
                # Check if numbers match perfectly
                if sql_count == expected_count:
                    print(f"✅ MATCH: Table '{table_name}' has exactly {sql_count} rows (Matches CSV).")
                else:
                    print(f"❌ MISMATCH: Table '{table_name}' has {sql_count} rows, but CSV had {expected_count}!")
                    
            except Exception as e:
                print(f"Could not verify table '{table_name}': {e}")

if __name__ == "__main__":
    load_and_verify_data()

--- STEP 1: LOADING DATASETS INTO SQLITE ---
Successfully loaded data/raw/01_fund_master.csv into table 'dim_fund'.
Successfully loaded cleaned_nav_history.csv into table 'fact_nav'.
Successfully loaded cleaned_investor_transactions.csv into table 'fact_transactions'.
Successfully loaded cleaned_scheme_performance.csv into table 'fact_performance'.

--- STEP 2: VERIFYING ROW COUNTS ---
✅ MATCH: Table 'dim_fund' has exactly 40 rows (Matches CSV).
✅ MATCH: Table 'fact_nav' has exactly 46000 rows (Matches CSV).
✅ MATCH: Table 'fact_transactions' has exactly 32778 rows (Matches CSV).
✅ MATCH: Table 'fact_performance' has exactly 40 rows (Matches CSV).


In [10]:
from sqlalchemy import create_engine, inspect

engine = create_engine('sqlite:///bluestock_mf.db')
inspector = inspect(engine)

for table in ['fact_transactions', 'fact_performance', 'fact_nav', 'dim_fund', 'dim_date']:
    cols = [col['name'] for col in inspector.get_columns(table)]
    print(f"{table}: {cols}")

fact_transactions: ['investor_id', 'transaction_date', 'amfi_code', 'transaction_type', 'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender', 'annual_income_lakh', 'payment_mode', 'kyc_status']
fact_performance: ['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct', 'morningstar_rating', 'risk_grade']
fact_nav: ['nav_id', 'amfi_code', 'nav_date', 'nav']
dim_fund: ['amfi_code', 'scheme_name', 'category', 'plan', 'expense_ratio_pct', 'risk_category']
dim_date: ['date', 'year', 'month', 'month_name', 'is_weekend']


In [12]:
import pandas as pd
from sqlalchemy import create_engine, text

engine = create_engine('sqlite:///bluestock_mf.db')

with engine.connect() as conn:

    q1 = pd.read_sql(text("""
        SELECT amfi_code, scheme_name, aum_crore
        FROM fact_performance
        ORDER BY aum_crore DESC LIMIT 5
    """), conn)
    print("1. Top 5 Funds by AUM\n", q1, "\n")

    q2 = pd.read_sql(text("""
        SELECT f.scheme_name, d.year, d.month_name, ROUND(AVG(n.nav), 2) AS avg_monthly_nav
        FROM fact_nav n
        JOIN fact_performance f ON n.amfi_code = f.amfi_code
        JOIN dim_date d ON n.nav_date = d.date
        GROUP BY f.scheme_name, d.year, d.month, d.month_name
        ORDER BY f.scheme_name, d.year, d.month LIMIT 10
    """), conn)
    print("2. Average NAV Per Month\n", q2, "\n")

    q3 = pd.read_sql(text("""
        SELECT state, COUNT(*) AS total_transactions,
               ROUND(SUM(amount_inr), 2) AS total_invested_amount
        FROM fact_transactions
        GROUP BY state
        ORDER BY total_invested_amount DESC
    """), conn)
    print("3. Total Transaction Volume by State\n", q3, "\n")

    q4 = pd.read_sql(text("""
        SELECT amfi_code, scheme_name, category, expense_ratio_pct
        FROM dim_fund
        WHERE expense_ratio_pct < 1.0
        ORDER BY expense_ratio_pct ASC
    """), conn)
    print("4. Low-Cost Funds (Expense Ratio < 1%)\n", q4, "\n")

    q5 = pd.read_sql(text("""
        SELECT scheme_name, category, morningstar_rating, return_3yr_pct
        FROM fact_performance
        WHERE morningstar_rating >= 4
        ORDER BY return_3yr_pct DESC
    """), conn)
    print("5. Highly Rated Funds with Top 3-Year Returns\n", q5, "\n")

    q6 = pd.read_sql(text("""
        SELECT transaction_type, COUNT(*) AS volume_count,
               ROUND(SUM(amount_inr), 2) AS total_inflow
        FROM fact_transactions
        GROUP BY transaction_type
    """), conn)
    print("6. SIP vs Lumpsum Transaction Breakdown\n", q6, "\n")

    q7 = pd.read_sql(text("""
        SELECT scheme_name, risk_grade, alpha, beta, sharpe_ratio
        FROM fact_performance
        WHERE alpha > 0.0 AND beta < 1.0
        ORDER BY alpha DESC
    """), conn)
    print("7. Risk vs Reward Profile\n", q7, "\n")

    q8 = pd.read_sql(text("""
        SELECT CASE WHEN d.is_weekend = 1 THEN 'Weekend' ELSE 'Weekday' END AS day_type,
               COUNT(*) AS total_transactions,
               ROUND(SUM(t.amount_inr), 2) AS total_value
        FROM fact_transactions t
        JOIN dim_date d ON t.transaction_date = d.date
        GROUP BY d.is_weekend
    """), conn)
    print("8. Weekends vs Weekdays\n", q8, "\n")

    q9 = pd.read_sql(text("""
        SELECT scheme_name, category, risk_grade, max_drawdown_pct
        FROM fact_performance
        ORDER BY max_drawdown_pct ASC
    """), conn)
    print("9. Most Volatile Schemes\n", q9, "\n")

    q10 = pd.read_sql(text("""
        SELECT age_group, gender, COUNT(*) AS total_investors,
               ROUND(SUM(amount_inr), 2) AS total_capital_allocated
        FROM fact_transactions
        WHERE kyc_status = 'Verified'
        GROUP BY age_group, gender
        ORDER BY total_capital_allocated DESC
    """), conn)
    print("10. Investor Demographics by Age Group\n", q10, "\n")

1. Top 5 Funds by AUM
    amfi_code                                        scheme_name  aum_crore
0     148568  Mirae Asset Emerging Bluechip Fund - Regular -...      49046
1     120842      Kotak Emerging Equity Fund - Regular - Growth      47469
2     118634     Nippon India Small Cap Fund - Regular - Growth      43630
3     149322         DSP Top 100 Equity Fund - Regular - Growth      41828
4     102886                UTI Mid Cap Fund - Regular - Growth      41728 

2. Average NAV Per Month
 Empty DataFrame
Columns: [scheme_name, year, month_name, avg_monthly_nav]
Index: [] 

3. Total Transaction Volume by State
              state  total_transactions  total_invested_amount
0           Punjab                2965            315780459.0
1       Tamil Nadu                2806            315177237.0
2   Madhya Pradesh                2931            308312493.0
3        Rajasthan                2577            298645822.0
4          Gujarat                2780            298358940.0
5  